# 06-01 优化器进阶：RMSProp 与 Adam 再理解

前面我们已经学过指数加权移动平均：

$$
v_t=\beta v_{t-1}+(1-\beta)x_t
$$

现在可以重新理解优化器。

SGD 只看当前梯度，Momentum 用 EMA 记住梯度方向，RMSProp 用 EMA 记住梯度大小，Adam 则把这两个思想合在一起。

## 1. 为什么 SGD 还不够

SGD 的更新公式是：

$$
\theta_t=\theta_{t-1}-\eta g_t
$$

其中：

$$
g_t=\nabla_{\theta}\mathcal{L}_t
$$

SGD 的问题是：所有参数共用同一个学习率 $\eta$。

但不同参数的梯度情况可能完全不同。

有的参数梯度经常很大，说明它变化很敏感；如果还用同样的大步子，可能震荡。

有的参数梯度经常很小，说明它更新很慢；如果还用同样的小步子，可能几乎不动。

所以问题变成：**能不能给不同参数自动调整不同的实际步幅？**

## 2. 自适应学习率的直觉

自适应学习率的想法是：不要让每个参数都用完全一样的步幅。

可以先用一句话理解：

```text
梯度经常大的参数，走小一点
梯度经常小的参数，走大一点
```

为什么？

如果某个方向非常陡，梯度很大，说明沿这个方向一点点变化就会让损失变化很多，所以应该谨慎一点。

如果某个方向很平，梯度很小，说明沿这个方向变化不明显，所以可以稍微大胆一点。

这个思想就是 AdaGrad、RMSProp、Adam 的共同出发点。

## 3. AdaGrad 先解决了什么

AdaGrad 的想法是：记录每个参数历史梯度平方的累积和。

设当前梯度是：

$$
g_t
$$

历史平方梯度累积为：

$$
s_t=s_{t-1}+g_t^2
$$

然后更新参数：

$$
\theta_t=\theta_{t-1}-\eta\frac{g_t}{\sqrt{s_t}+\epsilon}
$$

分母 $\sqrt{s_t}$ 表示这个参数过去梯度大不大。

如果某个参数历史梯度一直很大，那么 $s_t$ 会很大，分母变大，实际步幅变小。

这就实现了自适应步幅。

## 4. AdaGrad 的问题

AdaGrad 的问题也来自这个累积和：

$$
s_t=s_{t-1}+g_t^2
$$

$s_t$ 只会越来越大，不会变小。

训练到后面，分母可能越来越大：

$$
\sqrt{s_t}+\epsilon
$$

于是实际更新步幅越来越小，模型可能还没到好位置就走不动了。

所以 AdaGrad 的优点是能自动缩小经常出现大梯度的参数步幅；缺点是学习率可能过早衰减。

## 5. RMSProp 为什么出现

RMSProp 可以理解成对 AdaGrad 的改进。

AdaGrad 是把所有历史梯度平方一直加起来：

$$
s_t=s_{t-1}+g_t^2
$$

RMSProp 不想让很久以前的梯度一直影响现在，所以它用指数加权移动平均来记录梯度平方：

$$
s_t=\beta s_{t-1}+(1-\beta)g_t^2
$$

这句话的意思是：

```text
最近的梯度平方更重要
很久以前的梯度平方逐渐淡出
```

这就避免了 AdaGrad 分母无限增大的问题。

## 6. RMSProp 的更新公式怎么读

RMSProp 先计算梯度平方的 EMA：

$$
s_t=\beta s_{t-1}+(1-\beta)g_t^2
$$

再更新参数：

$$
\theta_t=\theta_{t-1}-\eta\frac{g_t}{\sqrt{s_t}+\epsilon}
$$

这个式子可以分成三块看。

第一，$g_t$ 决定当前更新方向。

第二，$s_t$ 记录最近一段时间梯度平方的平均大小。

第三，用 $\sqrt{s_t}+\epsilon$ 除一下，相当于自动调节每个参数的实际步幅。

如果某个参数最近梯度经常很大，$s_t$ 变大，步幅会被压小。

如果某个参数最近梯度经常很小，$s_t$ 较小，步幅不会被压得太厉害。

## 7. RMSProp 解决了什么

RMSProp 主要解决两个问题。

第一，它让不同参数有不同的实际学习率。

第二，它不像 AdaGrad 那样让历史梯度平方无限累积，而是更关注近期趋势。

所以 RMSProp 比 AdaGrad 更适合非平稳的深度学习训练过程。

这里的非平稳可以理解为：训练过程中，参数一直在变，损失曲面上的当前位置一直在变，早期梯度统计不应该永远主导后期训练。

## 8. Momentum 和 RMSProp 的区别

Momentum 和 RMSProp 都用到了 EMA，但记住的东西不一样。

Momentum 记住的是梯度本身：

$$
m_t=\beta m_{t-1}+(1-\beta)g_t
$$

它关心的是：最近一段时间大方向往哪里走。

RMSProp 记住的是梯度平方：

$$
s_t=\beta s_{t-1}+(1-\beta)g_t^2
$$

它关心的是：最近一段时间这个参数的梯度大不大。

所以：

```text
Momentum 解决方向抖动
RMSProp 解决不同参数步幅不一样的问题
```

## 9. Adam 是怎么组合二者的

Adam 可以理解成 Momentum 和 RMSProp 的结合。

它既记录梯度的一阶矩，也记录梯度平方的二阶矩。

一阶矩：

$$
m_t=\beta_1m_{t-1}+(1-\beta_1)g_t
$$

二阶矩：

$$
v_t=\beta_2v_{t-1}+(1-\beta_2)g_t^2
$$

$m_t$ 像 Momentum，表示方向趋势。

$v_t$ 像 RMSProp，表示梯度大小趋势。

Adam 的核心思想是：

```text
用 m_t 决定往哪里走
用 v_t 调整每个参数走多大步
```

## 10. Adam 为什么需要偏差修正

Adam 的 $m_t$ 和 $v_t$ 通常从 $0$ 开始：

$$
m_0=0,\quad v_0=0
$$

这会让前几步的移动平均偏小。

所以 Adam 会做偏差修正：

$$
\hat{m}_t=\frac{m_t}{1-\beta_1^t}
$$

$$
\hat{v}_t=\frac{v_t}{1-\beta_2^t}
$$

这和上一节指数加权移动平均里的偏差修正是同一个思想。

修正后的 Adam 更新是：

$$
\theta_t=\theta_{t-1}-\eta\frac{\hat{m}_t}{\sqrt{\hat{v}_t}+\epsilon}
$$

## 11. 为什么 Adam 常用默认参数

Adam 常见默认值是：

$$
\beta_1=0.9
$$

$$
\beta_2=0.999
$$

$\beta_1=0.9$ 表示一阶矩大约参考最近 $10$ 步左右的梯度方向。

因为：

$$
\frac{1}{1-0.9}=10
$$

$\beta_2=0.999$ 表示二阶矩参考更长时间的梯度平方趋势。

因为：

$$
\frac{1}{1-0.999}=1000
$$

直觉是：方向可以稍微灵活一点，但梯度大小的统计希望更稳定。

## 12. Adam 和 AdamW 的区别

AdamW 是 Adam 的一个常见改进版本。

它主要处理的是权重衰减的实现方式。

普通 Adam 中，如果把 $L_2$ 正则化直接加进损失函数，权重衰减会和 Adam 的自适应学习率混在一起。

AdamW 的思想是：把权重衰减从梯度更新里解耦出来，单独对权重做衰减。

可以粗略理解成：

```text
Adam：自适应更新和 L2 正则容易缠在一起
AdamW：自适应更新归自适应更新，权重衰减归权重衰减
```

现在很多 Transformer、预训练模型和深度学习项目更常用 AdamW。

入门阶段先记住：AdamW 是更适合配合权重衰减的 Adam 变体。

## 13. 优化器之间的逻辑关系

可以把这些优化器放在一条线里理解：

```text
SGD
-> Momentum：给梯度方向加记忆，减少抖动
-> AdaGrad：给每个参数自适应步幅
-> RMSProp：用 EMA 改进 AdaGrad，避免历史平方梯度无限累积
-> Adam：Momentum + RMSProp，并加入偏差修正
-> AdamW：Adam + 更合理的权重衰减
```

这样看，Adam 不是突然出现的复杂公式，而是一步步解决问题累积出来的结果。

## 14. 初学者怎么记

先不要把优化器当成 API 名字背。

可以这样记：

| 优化器 | 记忆方式 |
|---|---|
| SGD | 当前梯度告诉我往哪走 |
| Momentum | 不只看当前梯度，还保留过去方向 |
| AdaGrad | 梯度大的参数以后走小点 |
| RMSProp | 只看近期梯度大小，不让很久以前一直影响现在 |
| Adam | 既记方向，又调步幅 |
| AdamW | Adam，再把权重衰减处理得更干净 |

如果只是入门训练一个普通神经网络，Adam 或 AdamW 通常是比较友好的起点。

如果你想研究最终泛化表现，可以再尝试 SGD + Momentum。

## 15. 本节总结

这一节的逻辑链是：

```text
SGD 所有参数共用同一个学习率
-> 不同参数梯度大小不同，应该有不同实际步幅
-> AdaGrad 累积历史梯度平方，但分母会越来越大
-> RMSProp 用 EMA 记录近期梯度平方，避免无限累积
-> Momentum 用 EMA 记录梯度方向
-> Adam 同时使用方向 EMA 和平方梯度 EMA
-> AdamW 进一步把权重衰减解耦
```

先记住三个公式：

RMSProp：

$$
s_t=\beta s_{t-1}+(1-\beta)g_t^2
$$

Adam 一阶矩：

$$
m_t=\beta_1m_{t-1}+(1-\beta_1)g_t
$$

Adam 二阶矩：

$$
v_t=\beta_2v_{t-1}+(1-\beta_2)g_t^2
$$

到这里，优化器这条线就比较完整了。下一步可以继续进入学习率调度：为什么训练过程中学习率常常不是固定不变的。